# GenEuclidean v6 — ChemLM Decoder Suite: End-to-End A100 Experiment Notebook

Trains and evaluates the **`chemlm` / `chemlm_moe`** pocket- and property-conditioned
chemical language decoders end to end, then exports every table and figure as a single
downloadable bundle.

**Run it:** `Runtime > Change runtime type > A100 GPU`, then `Runtime > Run all`. Nothing
else needs editing. Every cell is idempotent and safe to re-run, and the long stage
checkpoints to Google Drive (if mounted) so a disconnect does not lose work.

### What this notebook establishes

| # | Experiment | Question it answers |
|---|---|---|
| E1 | Scale sweep `tiny`->`xl` | What does each preset cost in params, throughput and memory? |
| E2 | Stage-1 chemical prior | Does it generate **valid, unique, novel** molecules from real ZINC-250k? |
| E3 | Property conditioning | Do QED / MW / logP targets actually steer the output distribution? |
| E4 | Dense vs sparse MoE | Does top-k routing buy anything at equal width? |
| E5 | KV cache + CFG | Decoding speedup, and what guidance scale costs |
| E6 | Pocket conditioning | Does 3D pocket memory change what gets generated? |

### Read this before trusting a big number

Three findings from the 2025-2026 literature are built into the defaults here, because
they contradict the intuition that a bigger model is a better molecule generator:

1. **Scaling saturates early.** NovoMolGen (arXiv:2508.13408, ~30k experiments) finds
   goal-directed generation "largely saturates beyond approximately 5M parameters": the
   PMO sum moves only 16.48 -> 16.70 from 32M to 300M params, and docking top-5% moves
   -65.8 (5M) -> -66.2 (1B) kcal/mol, within noise across 200x the parameters. For
   SMILES the fitted parameter exponent is alpha=0.0171 against a data exponent
   beta=0.4299 (arXiv:2601.22757), so **data buys roughly 25x more than parameters**.
2. **Loss and sample quality can move in opposite directions.** Scaling 5M->32M on
   PubChem improved loss 54.1->48.1 bits/molecule while **FCD got worse, 2.8->5.1**.
   This notebook therefore reports sample metrics separately and never selects on loss.
3. **Memorization is the real failure mode.** GP-MoLFormer on 1.1B SMILES reports
   novelty 0.323 (68% of samples are verbatim training molecules), with ~45% of its
   corpus being duplicates; deduplicating lifted novelty to 0.390. **E2 deduplicates the
   corpus and measures novelty against it.**

`xl` (1.32B) is included so the saturation claim can be tested rather than assumed. The
recommended production preset is **`small` (28.6M)**.


## 1 - Hardware detection, A100 check and Drive sync


In [ ]:
#@title 1.1 Detect GPU, verify A100, optionally mount Drive
import json
import platform
import subprocess
import sys


MOUNT_DRIVE = True  #@param {type:"boolean"}


def sh(argv):
    # argv list, no shell: keeps quoting correct for paths containing spaces.
    try:
        return subprocess.run(
            argv, capture_output=True, text=True, timeout=120
        ).stdout.strip()
    except Exception as exc:
        return f"(failed: {exc})"


IN_COLAB = "google.colab" in sys.modules
print(f"Colab runtime : {IN_COLAB}")
print(f"Python        : {sys.version.split()[0]}  ({platform.platform()})")

GPU_NAME, GPU_MEM_GB = "none", 0.0
smi = sh(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])
if smi and "failed" not in smi and "," in smi:
    GPU_NAME = smi.split(",")[0].strip()
    GPU_MEM_GB = float(smi.split(",")[1].strip().split()[0]) / 1024.0
    print(f"GPU           : {GPU_NAME}  ({GPU_MEM_GB:.1f} GiB)")
else:
    print("GPU           : NONE DETECTED")

IS_A100 = "A100" in GPU_NAME
if not IS_A100:
    print("\n" + "-" * 66)
    print(f"This runtime is not an A100 (detected: {GPU_NAME or 'CPU only'}).")
    print("")
    print("Select one with:  Runtime > Change runtime type > A100 GPU > Save")
    print("(A100 requires Colab Pro / Pro+ compute units.)")
    print("")
    print("The notebook still runs correctly on T4, L4, V100 or CPU and will")
    print("shrink the experiment budget automatically. Only the xl preset")
    print("(1.32B params) genuinely needs an A100 40/80GB.")
    print("-" * 66 + "\n")
else:
    print(f"A100 confirmed -> full experiment budget enabled ({GPU_MEM_GB:.0f} GiB)")

DRIVE_DIR = None
if IN_COLAB and MOUNT_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        DRIVE_DIR = "/content/drive/MyDrive/geneuclidean_v6"
        import os

        os.makedirs(DRIVE_DIR, exist_ok=True)
        print(f"Drive         : {DRIVE_DIR}")
    except Exception as exc:
        print(f"Drive mount skipped: {exc}")

HW = {
    "in_colab": IN_COLAB,
    "gpu": GPU_NAME,
    "gpu_mem_gb": GPU_MEM_GB,
    "is_a100": IS_A100,
    "drive_dir": DRIVE_DIR,
}
print("\n" + json.dumps(HW, indent=2))


## 2 - Experiment configuration


In [ ]:
#@title 2.1 Master controls
#@markdown Defaults target a single A100 and finish in roughly 2-3 hours.

REPO_URL = "https://github.com/daniil-emtsev/geneuclidean"  #@param {type:"string"}
REPO_BRANCH = "feat/molecular-llm-decoder-suite"  #@param {type:"string"}

#@markdown ---
#@markdown **Which experiments to run**
RUN_E1_SCALES = True  #@param {type:"boolean"}
RUN_E2_PRIOR = True  #@param {type:"boolean"}
RUN_E3_PROPERTY = True  #@param {type:"boolean"}
RUN_E4_MOE = True  #@param {type:"boolean"}
RUN_E5_DECODING = True  #@param {type:"boolean"}
RUN_E6_POCKET = True  #@param {type:"boolean"}

#@markdown ---
#@markdown **Stage-1 chemical prior (E2) - the main training job**
TRAIN_SCALE = "small"  #@param ["tiny", "small", "base", "large", "xl"]
TRAIN_STEPS = 12000  #@param {type:"integer"}
BATCH_SIZE = 128  #@param {type:"integer"}
SEQ_LEN = 96  #@param {type:"integer"}
LEARNING_RATE = 3e-4  #@param {type:"number"}
NUM_SAMPLES = 2048  #@param {type:"integer"}
CORPUS_LIMIT = 250000  #@param {type:"integer"}
DEDUPE_CORPUS = True  #@param {type:"boolean"}

#@markdown ---
#@markdown **Scale sweep (E1)**
SWEEP_SCALES = "tiny,small,base,large,xl"  #@param {type:"string"}

#@markdown ---
#@markdown **Pocket corpus (E6)**
CROSSDOCKED_TIER = "smoke"  #@param ["skip", "smoke", "medium"]

SEED = 42  #@param {type:"integer"}

# Shrink the budget automatically without an A100 so Run-all still completes.
if not IS_A100:
    _order = ["tiny", "small", "base", "large", "xl"]
    _cap = "small" if GPU_MEM_GB < 20 else "base"
    if _order.index(TRAIN_SCALE) > _order.index(_cap):
        print(f"[auto] TRAIN_SCALE {TRAIN_SCALE} -> {_cap} (no A100)")
        TRAIN_SCALE = _cap
    SWEEP_SCALES = ",".join(_order[: _order.index(_cap) + 1])
    TRAIN_STEPS = min(TRAIN_STEPS, 3000)
    BATCH_SIZE = min(BATCH_SIZE, 32)
    NUM_SAMPLES = min(NUM_SAMPLES, 512)
    print(f"[auto] budget: steps={TRAIN_STEPS} batch={BATCH_SIZE} sweep={SWEEP_SCALES}")

CFG = {
    "repo_url": REPO_URL,
    "repo_branch": REPO_BRANCH,
    "train_scale": TRAIN_SCALE,
    "train_steps": TRAIN_STEPS,
    "batch_size": BATCH_SIZE,
    "seq_len": SEQ_LEN,
    "lr": LEARNING_RATE,
    "num_samples": NUM_SAMPLES,
    "corpus_limit": CORPUS_LIMIT,
    "dedupe": DEDUPE_CORPUS,
    "sweep_scales": SWEEP_SCALES,
    "crossdocked_tier": CROSSDOCKED_TIER,
    "seed": SEED,
}
print(json.dumps(CFG, indent=2))


## 3 - Environment


In [ ]:
#@title 3.1 Clone the repo and install (uv, with a pip fallback)
import os
import subprocess
import sys

WORK = "/content" if IN_COLAB else os.getcwd()
REPO = os.path.join(WORK, "geneuclidean")


def run(argv, check=True, cwd=None):
    # argv lists only, and cwd= instead of "cd X && ...": paths may contain spaces.
    print("$ " + " ".join(argv))
    rc = subprocess.run(argv, cwd=cwd, text=True).returncode
    if check and rc != 0:
        raise RuntimeError(f"command failed ({rc}): {' '.join(argv)}")
    return rc


if IN_COLAB:
    if not os.path.isdir(os.path.join(REPO, ".git")):
        run(
            ["git", "clone", "--depth", "1", "--branch", CFG["repo_branch"],
             CFG["repo_url"], REPO],
            check=False,
        )
    if not os.path.isdir(os.path.join(REPO, ".git")):
        raise RuntimeError(
            f"Could not clone {CFG['repo_url']} @ {CFG['repo_branch']}.\n"
            "If the repository is private, upload it instead:\n"
            "  from google.colab import files; files.upload()   # a .tar.gz of the repo\n"
            "then extract it to /content/geneuclidean and re-run this cell."
        )
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=False)
    rc = run(
        [sys.executable, "-m", "uv", "pip", "install", "--system", "-q", "-e", ".[dev]"],
        check=False, cwd=REPO,
    )
    if rc != 0:
        print("uv failed; falling back to pip")
        run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"],
            check=False, cwd=REPO)
else:
    REPO = os.getcwd()
    print(f"Local run, using {REPO}")

if os.path.join(REPO, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO, "src"))
os.chdir(REPO)

import torch

print(f"\ntorch {torch.__version__}  cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("TF32 enabled for matmul and cuDNN")

import geneuclidean as ge

print(f"geneuclidean {getattr(ge, '__version__', '?')}")
print(f"decoders: {ge.list_decoders()}")
assert "chemlm" in ge.list_decoders(), "chemlm decoder not registered"
assert "chemlm_moe" in ge.list_decoders(), "chemlm_moe decoder not registered"
print("OK")


## 4 - Correctness gate (fail fast)

Runs the decoder and corpus test suites **before** any GPU time is spent. If the
architecture is broken this stops here, rather than after a three-hour training run.


In [ ]:
#@title 4.1 Run the chemlm + corpus test suites
import subprocess
import sys

# argv list, never shell=True: sys.executable can contain spaces (common on macOS),
# which silently breaks an interpolated shell command.
argv = [
    sys.executable, "-m", "pytest", "-q", "-p", "no:randomly",
    "tests/test_chemlm_decoders.py", "tests/test_pocket_corpora.py",
]
print("$ " + " ".join(argv) + "\n")
proc = subprocess.run(argv, text=True, capture_output=True)
print(proc.stdout[-4000:])
if proc.returncode != 0:
    print(proc.stderr[-3000:])
    raise RuntimeError("Correctness gate FAILED - not spending GPU time on a broken build.")
print("\nCorrectness gate PASSED")


In [ ]:
#@title 4.2 Device-level numerical checks on this GPU
import torch

from geneuclidean.models.decoders.chemlm import ChemLMDecoder

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(CFG["seed"])

dec = (
    ChemLMDecoder(
        vocab_size=64, d_model=64, n_layers=2, n_heads=4, n_kv_heads=2,
        d_ff=128, max_seq_len=64, dropout=0.0, num_properties=3,
    )
    .to(dev)
    .eval()
)
ids = torch.randint(4, 64, (2, 16), device=dev)
mem = torch.randn(2, 12, 64, device=dev)
msk = torch.ones(2, 12, dtype=torch.bool, device=dev)

with torch.no_grad():
    full, _ = dec(ids, memory=mem, memory_mask=msk)

    cache, steps = None, []
    for t in range(ids.shape[1]):
        out, cache = dec.forward_cached(
            ids[:, t : t + 1], memory=mem, memory_mask=msk, kv_cache=cache
        )
        steps.append(out)
    cached = torch.cat(steps, 1)

    junk = torch.randn(2, 5, 64, device=dev) * 50.0
    pad_mask = torch.zeros(2, 5, dtype=torch.bool, device=dev)
    padded, _ = dec(
        ids,
        memory=torch.cat([mem, junk], 1),
        memory_mask=torch.cat([msk, pad_mask], 1),
    )

    # Causal masking: perturbing tokens >= 8 must not move logits < 8.
    b = ids.clone()
    b[:, 8:] = (ids[:, 8:] + 5) % 63 + 1
    lb, _ = dec(b, memory=mem, memory_mask=msk)

checks = {
    "kv_cache_parity": float((cached - full).abs().max()),
    "padded_pocket_leak": float((padded - full).abs().max()),
    "causal_leakage": float((full[:, :8] - lb[:, :8]).abs().max()),
    "logits_finite": bool(torch.isfinite(full).all()),
}
for k, v in checks.items():
    print(f"  {k:<22} {v}")

assert checks["kv_cache_parity"] < 2e-4, checks
assert checks["padded_pocket_leak"] < 1e-5, checks
assert checks["causal_leakage"] == 0.0, f"dense decoder leaked: {checks}"
assert checks["logits_finite"]
print("\nDevice numerical checks PASSED")


## 5 - Data: ZINC-250k chemical prior and CrossDocked2020 pockets


In [ ]:
#@title 5.1 Fetch ZINC-250k, deduplicate, build the corpus
import csv
import json
import os
import time
from pathlib import Path

from geneuclidean.data.download import download_zinc250k

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

zinc_csv = Path("data/raw/zinc250k.csv")
if not zinc_csv.exists():
    print("Downloading ZINC-250k ...")
    download_zinc250k(Path("data/raw"))
print(f"ZINC csv: {zinc_csv} ({zinc_csv.stat().st_size / 1e6:.1f} MB)")

raw = []
with zinc_csv.open() as fh:
    for row in csv.DictReader(fh):
        smi = (row.get("smiles") or row.get("SMILES") or "").strip()
        if smi:
            raw.append(smi)
        if len(raw) >= CFG["corpus_limit"]:
            break
print(f"raw molecules: {len(raw)}")

# Deduplicate on canonical SMILES. GP-MoLFormer trained on a corpus that was ~45%
# duplicates and reported novelty 0.323; deduplicating lifted it to 0.390.
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")

t0 = time.time()
if CFG["dedupe"]:
    seen, corpus = set(), []
    for smi in raw:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        can = Chem.MolToSmiles(mol)
        if can not in seen:
            seen.add(can)
            corpus.append(can)
    dup_rate = 1.0 - len(corpus) / max(1, len(raw))
    print(
        f"deduplicated: {len(raw)} -> {len(corpus)} "
        f"({dup_rate:.2%} removed, {time.time() - t0:.0f}s)"
    )
else:
    corpus, dup_rate = raw, None

Path("data/processed/corpus_dedup.smi").write_text("\n".join(corpus) + "\n")
DATA_STATS = {"raw": len(raw), "corpus": len(corpus), "duplicate_rate": dup_rate}
print(json.dumps(DATA_STATS, indent=2))


In [ ]:
#@title 5.2 CrossDocked2020 index + leakage-safe split (CC0, no login)
import json
from pathlib import Path

from geneuclidean.data.download import download_file
from geneuclidean.data.pocket_corpora import (
    build_rmsd_filtered_pairs,
    crossdocked_download_plan,
    iter_crossdocked_types,
    make_pocket_disjoint_split,
    summarize_pairs,
    write_split_manifest,
)

CROSS = {"skipped": True}
if CFG["crossdocked_tier"] != "skip":
    plan = crossdocked_download_plan(CFG["crossdocked_tier"])
    for v in plan:
        print(f"{v.name}: {v.approx_gib:.3f} GiB  {v.url}")

    dest_dir = Path("data/raw/crossdocked")
    dest_dir.mkdir(parents=True, exist_ok=True)

    got = []
    for v in plan:
        dest = dest_dir / v.filename
        if not dest.exists():
            try:
                download_file(v.url, dest, timeout=1800)
            except Exception as exc:
                print(f"  download failed ({v.name}): {exc}")
                continue
        if dest.exists():
            got.append(dest)
            print(f"  have {dest.name} ({dest.stat().st_size / 1e6:.1f} MB)")

    archive = next((p for p in got if "types" in p.name), None)
    if archive is not None:
        pairs = list(iter_crossdocked_types(archive))
        print(f"\nparsed {len(pairs)} pose records")
        # Keep ALL qualifying pairs: the canonical pipeline draws only 100k of 184,057.
        kept = build_rmsd_filtered_pairs(pairs, rmsd_threshold=1.0)
        print(f"RMSD<=1A, deduped: {len(kept)} pairs")
        print(json.dumps(summarize_pairs(kept), indent=2))

        n_groups = len({p.pocket_group for p in kept})
        if n_groups >= 12:
            split = make_pocket_disjoint_split(
                kept,
                val_fraction=0.05,
                test_pocket_count=min(100, max(2, n_groups // 10)),
                seed=CFG["seed"],
            )
            split.assert_disjoint()
            write_split_manifest(split, "data/processed/crossdocked_split.json")
            CROSS = {
                "skipped": False,
                "pairs": len(kept),
                "sizes": split.sizes,
                "summary": summarize_pairs(kept),
            }
            print(f"\npocket-disjoint split: {split.sizes}")
        else:
            CROSS = {
                "skipped": False,
                "pairs": len(kept),
                "note": f"only {n_groups} pocket groups; too few to split",
            }
print(json.dumps(CROSS, indent=2)[:1500])


## 6 - E1: scale sweep on this GPU


In [ ]:
#@title 6.1 Params, throughput and memory for every preset
import gc
import json
import statistics
import time

import torch
import torch.nn.functional as F

from geneuclidean.models.decoders.chemlm import SCALE_PRESETS, ChemLMDecoder

E1 = []
if RUN_E1_SCALES:
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    B = max(8, CFG["batch_size"] // 4)
    T = CFG["seq_len"]
    V = 2362  # GP-MoLFormer vocabulary size

    for name in [s.strip() for s in CFG["sweep_scales"].split(",") if s.strip()]:
        preset = SCALE_PRESETS[name]
        dec = opt = ids = None
        try:
            if dev.type == "cuda":
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
            dec = ChemLMDecoder(
                vocab_size=V, d_model=preset.d_model, n_layers=preset.n_layers,
                n_heads=preset.n_heads, n_kv_heads=preset.n_kv_heads,
                d_ff=preset.d_ff, max_seq_len=T, dropout=0.0, use_cross_attn=False,
            ).to(dev)
            params_m = dec.num_parameters() / 1e6
            ids = torch.randint(4, V, (B, T), device=dev)
            opt = torch.optim.AdamW(dec.parameters(), lr=1e-4)

            def one_step():
                logits, _ = dec(ids)
                loss = F.cross_entropy(
                    logits[:, :-1].reshape(-1, V).float(), ids[:, 1:].reshape(-1)
                )
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()

            dec.train()
            for _ in range(2):
                one_step()
            if dev.type == "cuda":
                torch.cuda.synchronize()

            samples = []
            for _ in range(5):
                t0 = time.perf_counter()
                one_step()
                if dev.type == "cuda":
                    torch.cuda.synchronize()
                samples.append(time.perf_counter() - t0)
            step_s = statistics.median(samples)
            peak = torch.cuda.max_memory_allocated() / 2**30 if dev.type == "cuda" else 0.0

            row = {
                "scale": name,
                "params_m": round(params_m, 2),
                "estimated_params_m": round(
                    preset.estimated_params(V, include_cross_attn=False) / 1e6, 2
                ),
                "d_model": preset.d_model,
                "n_layers": preset.n_layers,
                "batch": B,
                "ms_per_step": round(step_s * 1000, 1),
                "tokens_per_s": round(B * T / step_s),
                "peak_gib": round(peak, 2),
            }
            E1.append(row)
            print(
                f"  {name:<6} {params_m:>8.1f}M  {row['ms_per_step']:>8.1f} ms  "
                f"{row['tokens_per_s']:>9} tok/s  {peak:>5.2f} GiB"
            )
        except torch.cuda.OutOfMemoryError:
            print(f"  {name:<6} OOM at batch {B} - skipped")
            E1.append({"scale": name, "oom": True, "batch": B})
        finally:
            del dec, opt, ids
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
print(json.dumps(E1, indent=2))


## 7 - E2: Stage-1 chemical prior (the main training job)

Trains an unconditional chemical prior on the deduplicated corpus, then samples with
KV-cached decoding and scores **validity, uniqueness, novelty, QED**. Novelty is measured
against the deduplicated training set, which is the number that actually matters.


In [ ]:
#@title 7.1 Train the chemical prior and score generated chemistry
import json
import math
import random
import statistics
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from rdkit import Chem, RDLogger
from rdkit.Chem import QED

from geneuclidean.data.tokenizer import MolecularTokenizer
from geneuclidean.models.decoders.chemlm import SCALE_PRESETS, ChemLMDecoder

RDLogger.DisableLog("rdApp.*")

E2 = {}
if RUN_E2_PRIOR:
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    corpus = Path("data/processed/corpus_dedup.smi").read_text().split()
    print(f"corpus: {len(corpus)} molecules")

    tok = MolecularTokenizer(representation="smiles").fit(corpus[:50000], min_freq=1)
    V, T = tok.vocab_size, CFG["seq_len"]
    print(f"vocab: {V}")

    enc = [tok.encode(s, max_len=T) for s in corpus]
    enc = [e for e in enc if len(e) >= 4]

    preset = SCALE_PRESETS[CFG["train_scale"]]
    dec = ChemLMDecoder(
        vocab_size=V, d_model=preset.d_model, n_layers=preset.n_layers,
        n_heads=preset.n_heads, n_kv_heads=preset.n_kv_heads, d_ff=preset.d_ff,
        max_seq_len=T, dropout=0.1, use_cross_attn=False,
    ).to(dev)
    print(f"{CFG['train_scale']}: {dec.num_parameters() / 1e6:.1f}M params")

    opt = torch.optim.AdamW(
        dec.parameters(), lr=CFG["lr"], weight_decay=0.01, betas=(0.9, 0.95)
    )
    steps = CFG["train_steps"]
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=CFG["lr"], total_steps=steps, pct_start=0.05
    )
    amp_dtype = torch.bfloat16 if dev.type == "cuda" else torch.float32
    rng = random.Random(CFG["seed"])
    B = CFG["batch_size"]

    def make_batch():
        picked = [rng.choice(enc) for _ in range(B)]
        width = max(len(x) for x in picked)
        return torch.tensor(
            [x + [tok.PAD_ID] * (width - len(x)) for x in picked],
            dtype=torch.long, device=dev,
        )

    dec.train()
    losses = []
    t0 = time.time()
    for step in range(steps):
        ids = make_batch()
        with torch.autocast(device_type=dev.type, dtype=amp_dtype, enabled=dev.type == "cuda"):
            logits, _ = dec(ids)
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, V).float(),
                ids[:, 1:].reshape(-1),
                ignore_index=tok.PAD_ID,
            )
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(dec.parameters(), 1.0)
        opt.step()
        sched.step()
        losses.append(float(loss.detach()))
        if (step + 1) % max(1, steps // 20) == 0:
            el = time.time() - t0
            print(
                f"  step {step + 1:>6}/{steps}  loss {statistics.mean(losses[-50:]):.4f}  "
                f"{el:.0f}s  {(step + 1) * B * ids.shape[1] / el:.0f} tok/s"
            )
    train_s = time.time() - t0

    if DRIVE_DIR:
        ckpt = Path(DRIVE_DIR) / "chemlm_prior.pt"
        torch.save({"model_state_dict": dec.state_dict(), "vocab": tok.tokens}, ckpt)
        print(f"checkpoint -> {ckpt}")

    dec.eval()
    t0 = time.time()
    toks = dec.generate(
        batch_size=CFG["num_samples"], memory=None, max_new_tokens=T - 1,
        bos_id=tok.BOS_ID, eos_id=tok.EOS_ID, temperature=1.0, top_p=0.95,
        seed=CFG["seed"], device=dev,
    )
    sample_s = time.time() - t0

    decoded = [tok.decode(r.tolist()) for r in toks.cpu()]
    valid = [(s, Chem.MolFromSmiles(s)) for s in decoded if s]
    valid = [(s, m) for s, m in valid if m is not None]
    canon = {Chem.MolToSmiles(m) for _, m in valid}
    train_set = set(corpus)

    qeds = []
    for _, mol in valid:
        try:
            qeds.append(float(QED.qed(mol)))
        except Exception:
            pass

    mean_loss = statistics.mean(losses[-50:])
    E2 = {
        "scale": CFG["train_scale"],
        "params_m": round(dec.num_parameters() / 1e6, 2),
        "vocab": V,
        "corpus": len(enc),
        "steps": steps,
        "batch": B,
        "train_seconds": round(train_s, 1),
        "tokens_per_s": round(steps * B * T / train_s),
        "first_loss": round(statistics.mean(losses[:20]), 4),
        "final_loss": round(mean_loss, 4),
        "final_ppl": round(math.exp(min(20.0, mean_loss)), 2),
        "n_sampled": CFG["num_samples"],
        "sample_seconds": round(sample_s, 2),
        "validity": round(len(valid) / CFG["num_samples"], 4),
        "uniqueness": round(len(canon) / max(1, len(valid)), 4),
        "novelty": round(len(canon - train_set) / max(1, len(canon)), 4),
        "mean_qed": round(statistics.mean(qeds), 4) if qeds else 0.0,
        "examples": [s for s, _ in valid[:15]],
    }
    print("\n" + json.dumps({k: v for k, v in E2.items() if k != "examples"}, indent=2))
    print("\nexamples:")
    for s in E2["examples"]:
        print("  ", s)


## 8 - E3: property-conditioned generation

Trains on `(SMILES, QED, MolWt, LogP)` with per-property `unknown` dropout, then samples
with deliberately opposed targets. A working conditioner must shift the realised property
distribution in the requested direction, so the reported `*_separation` values
(`mean(high target) - mean(low target)`) must be positive.

Note: the Fourier-feature value encoding is **novel and not externally validated** - no
published chemical LM uses it. The `unknown`-dropout / classifier-free mechanism around it
*is* well supported: MolGPT-style prefix conditioning on 7 joint properties collapses to
25.7% validity, while a CFG-trained model holds 100%.


In [ ]:
#@title 8.1 Train a property-conditioned decoder and test steering
import json
import subprocess
import sys
from pathlib import Path

E3 = {}
if RUN_E3_PROPERTY:
    out_json = Path("results/tables/chemlm_notebook_tier4.json")
    argv = [
        sys.executable, "scripts/run_chemlm_decoder_benchmark.py",
        "--skip-tier3", "--device", "auto",
        "--train-scale", str(CFG["train_scale"]),
        "--train-steps", str(max(600, CFG["train_steps"] // 6)),
        "--batch-size", str(CFG["batch_size"]),
        "--seq-len", str(CFG["seq_len"]),
        "--num-samples", str(CFG["num_samples"]),
        "--corpus-limit", str(min(60000, CFG["corpus_limit"])),
        "--smiles", "data/processed/corpus_dedup.smi",
        "--iters", "3", "--out", str(out_json),
    ]
    print("$ " + " ".join(argv) + "\n")
    subprocess.run(argv, text=True)
    if out_json.exists():
        payload = json.loads(out_json.read_text())
        E3 = payload.get("tier4_property_conditioning", {})
        E3["tier2_correctness"] = payload.get("tier2_correctness", {})
        print(json.dumps(E3, indent=2)[:2500])
    else:
        print("tier4 produced no output")


## 9 - E4: dense vs sparse Mixture-of-Experts, and E5: decoding


In [ ]:
#@title 9.1 Dense vs MoE at equal width; KV-cache speedup; CFG sweep
import gc
import json
import statistics
import time

import torch
import torch.nn.functional as F

from geneuclidean.models.decoders.chemlm import SCALE_PRESETS, ChemLMDecoder

E4, E5 = {}, {}
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
preset = SCALE_PRESETS[CFG["train_scale"]]
V, T = 2362, CFG["seq_len"]
B = max(8, CFG["batch_size"] // 4)

if RUN_E4_MOE:
    rows = []
    variants = (
        ("dense", {}),
        ("moe_8x2", {"use_moe": True, "num_experts": 8, "moe_top_k": 2}),
    )
    for tag, kwargs in variants:
        dec = ChemLMDecoder(
            vocab_size=V, d_model=preset.d_model, n_layers=preset.n_layers,
            n_heads=preset.n_heads, n_kv_heads=preset.n_kv_heads, d_ff=preset.d_ff,
            max_seq_len=T, dropout=0.0, use_cross_attn=False, **kwargs,
        ).to(dev)
        ids = torch.randint(4, V, (B, T), device=dev)
        opt = torch.optim.AdamW(dec.parameters(), lr=1e-4)

        def one_step():
            logits, _ = dec(ids)
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, V).float(), ids[:, 1:].reshape(-1)
            ) + 0.01 * dec.aux_loss()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

        dec.train()
        for _ in range(3):
            one_step()
        if dev.type == "cuda":
            torch.cuda.synchronize()

        samples = []
        for _ in range(5):
            t0 = time.perf_counter()
            one_step()
            if dev.type == "cuda":
                torch.cuda.synchronize()
            samples.append(time.perf_counter() - t0)

        rows.append({
            "variant": tag,
            "params_m": round(dec.num_parameters() / 1e6, 2),
            "ms_per_step": round(statistics.median(samples) * 1000, 1),
            "aux_loss": round(float(dec.aux_loss().detach()), 4),
        })
        print(
            f"  {tag:<9} {rows[-1]['params_m']:>8.1f}M  "
            f"{rows[-1]['ms_per_step']:>8.1f} ms  aux={rows[-1]['aux_loss']}"
        )
        del dec, opt, ids
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    E4 = {"rows": rows}
    if len(rows) == 2:
        E4["param_ratio"] = round(rows[1]["params_m"] / rows[0]["params_m"], 2)
        E4["step_time_ratio"] = round(rows[1]["ms_per_step"] / rows[0]["ms_per_step"], 2)
        print(
            f"\nMoE gives {E4['param_ratio']}x the parameters "
            f"at {E4['step_time_ratio']}x the step time"
        )

if RUN_E5_DECODING:
    dec = (
        ChemLMDecoder(
            vocab_size=V, d_model=256, n_layers=6, n_heads=8, n_kv_heads=2,
            d_ff=688, max_seq_len=T, dropout=0.0,
        )
        .to(dev)
        .eval()
    )
    mem = torch.randn(64, 64, 256, device=dev)
    msk = torch.ones(64, 64, dtype=torch.bool, device=dev)

    t0 = time.perf_counter()
    dec.generate(
        batch_size=64, memory=mem, memory_mask=msk, max_new_tokens=64,
        bos_id=1, eos_id=2, temperature=1.0, seed=0, device=dev,
    )
    if dev.type == "cuda":
        torch.cuda.synchronize()
    cached_s = time.perf_counter() - t0

    # Uncached reference: a full forward pass at every step.
    t0 = time.perf_counter()
    with torch.no_grad():
        toks = torch.full((64, 1), 1, dtype=torch.long, device=dev)
        for _ in range(64):
            logits, _ = dec(toks, memory=mem, memory_mask=msk)
            nxt = torch.multinomial(torch.softmax(logits[:, -1].float(), -1), 1)
            toks = torch.cat([toks, nxt], 1)
    if dev.type == "cuda":
        torch.cuda.synchronize()
    uncached_s = time.perf_counter() - t0

    cfg_rows = []
    for w in (1.0, 1.5, 2.0, 3.0):
        t0 = time.perf_counter()
        out = dec.generate(
            batch_size=32, memory=mem[:32], memory_mask=msk[:32], max_new_tokens=48,
            bos_id=1, eos_id=2, temperature=1.0, cfg_scale=w, seed=0, device=dev,
        )
        if dev.type == "cuda":
            torch.cuda.synchronize()
        cfg_rows.append({
            "cfg_scale": w,
            "seconds": round(time.perf_counter() - t0, 2),
            "mean_len": round(float((out != 2).float().sum(1).mean()), 1),
        })
        print(
            f"  cfg={w:<4} {cfg_rows[-1]['seconds']:>6.2f}s  "
            f"mean_len={cfg_rows[-1]['mean_len']}"
        )

    E5 = {
        "kv_cached_seconds": round(cached_s, 2),
        "uncached_seconds": round(uncached_s, 2),
        "speedup": round(uncached_s / max(cached_s, 1e-6), 2),
        "cfg_sweep": cfg_rows,
    }
    print(f"\nKV cache speedup: {E5['speedup']}x ({uncached_s:.2f}s -> {cached_s:.2f}s)")
    del dec
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(json.dumps({"E4": E4, "E5": E5}, indent=2)[:2000])


## 10 - E6: pocket-conditioned generation


In [ ]:
#@title 10.1 Does 3D pocket memory change what gets generated?
import json

import torch
import torch.nn.functional as F

from geneuclidean.models.decoders.chemlm import ChemLMDecoder

E6 = {}
if RUN_E6_POCKET:
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(CFG["seed"])
    dec = (
        ChemLMDecoder(
            vocab_size=2362, d_model=256, n_layers=6, n_heads=8, n_kv_heads=2,
            d_ff=688, max_seq_len=CFG["seq_len"], dropout=0.0,
            use_cross_attn=True, cross_attn_gate_init=1.0,
        )
        .to(dev)
        .eval()
    )
    ids = torch.randint(4, 2362, (8, 24), device=dev)
    msk = torch.ones(8, 64, dtype=torch.bool, device=dev)

    with torch.no_grad():
        cond, attn = dec(
            ids, memory=torch.randn(8, 64, 256, device=dev), memory_mask=msk,
            return_attn=True,
        )
        other, _ = dec(
            ids, memory=torch.randn(8, 64, 256, device=dev) * 3.0, memory_mask=msk
        )
        uncond, _ = dec(ids, memory=None)

        lp_c = F.log_softmax(cond.float(), -1)
        lp_u = F.log_softmax(uncond.float(), -1)
        kl = float((lp_c.exp() * (lp_c - lp_u)).sum(-1).mean())

    E6 = {
        "pocket_sensitivity": round(float((cond - other).abs().max()), 6),
        "conditional_vs_unconditional_kl": round(kl, 6),
        "attn_shape": list(attn.shape) if attn is not None else None,
        "attn_rows_sum_to_one": (
            round(float(attn.sum(-1).mean()), 4) if attn is not None else None
        ),
    }
    for k, v in E6.items():
        print(f"  {k:<34} {v}")
    assert E6["pocket_sensitivity"] > 1e-5, "pocket conditioning is inert"
    assert E6["conditional_vs_unconditional_kl"] > 0.0, "pocket adds no information"
    print("\nPocket conditioning is active")
print(json.dumps(E6, indent=2))


## 11 - Results, figures and downloadable bundle


In [ ]:
#@title 11.1 Aggregate every experiment into one report
import json
import platform
import sys
import time
from pathlib import Path

import torch

REPORT = {
    "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "hardware": HW,
    "config": CFG,
    "torch": str(torch.__version__),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "data": DATA_STATS,
    "crossdocked": CROSS,
    "E1_scale_sweep": E1,
    "E2_chemical_prior": E2,
    "E3_property_conditioning": E3,
    "E4_moe": E4,
    "E5_decoding": E5,
    "E6_pocket": E6,
}
Path("results/tables").mkdir(parents=True, exist_ok=True)
out = Path("results/tables/chemlm_v6_notebook_report.json")
out.write_text(json.dumps(REPORT, indent=2, default=str) + "\n")
print(f"wrote {out}\n")


def row(key, value):
    print(f"  {key:<34} {value}")


print("=== HEADLINE ===")
if E2:
    row("prior scale / params", f"{E2['scale']} / {E2['params_m']}M")
    row("final loss / perplexity", f"{E2['final_loss']} / {E2['final_ppl']}")
    row("validity", E2["validity"])
    row("uniqueness", E2["uniqueness"])
    row("novelty (vs dedup corpus)", E2["novelty"])
    row("mean QED", E2["mean_qed"])
if E5:
    row("KV-cache speedup", f"{E5['speedup']}x")
if E4 and "param_ratio" in E4:
    row("MoE params / step-time", f"{E4['param_ratio']}x / {E4['step_time_ratio']}x")
for prop in ("qed", "mol_wt", "log_p"):
    if isinstance(E3, dict) and f"{prop}_separation" in E3:
        row(f"{prop} steering separation", E3[f"{prop}_separation"])


In [ ]:
#@title 11.2 Figures
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

Path("results/figures").mkdir(parents=True, exist_ok=True)
made = []

rows = [r for r in E1 if not r.get("oom")]
if rows:
    fig, ax = plt.subplots(1, 2, figsize=(11, 4), dpi=150)
    names = [r["scale"] for r in rows]
    ax[0].bar(names, [r["params_m"] for r in rows], color="#4C72B0")
    ax[0].set_yscale("log")
    ax[0].set_ylabel("parameters (M)")
    ax[0].set_title("Capacity by preset")
    ax[1].plot(names, [r["tokens_per_s"] for r in rows], "o-", color="#C44E52")
    ax[1].set_ylabel("tokens / s")
    ax[1].set_yscale("log")
    ax[1].set_title(f"Training throughput ({HW['gpu'] or 'CPU'})")
    for a in ax:
        a.grid(alpha=0.3)
    fig.tight_layout()
    path = "results/figures/e1_scale_sweep.png"
    fig.savefig(path)
    made.append(path)
    plt.close(fig)

if E2:
    fig, ax = plt.subplots(figsize=(6, 4), dpi=150)
    keys = ["validity", "uniqueness", "novelty", "mean_qed"]
    ax.bar(keys, [E2[k] for k in keys], color=["#55A868", "#4C72B0", "#C44E52", "#8172B2"])
    ax.set_ylim(0, 1.05)
    ax.set_title(f"Generated chemistry - {E2['scale']} ({E2['params_m']}M)")
    for i, k in enumerate(keys):
        ax.text(i, E2[k] + 0.02, f"{E2[k]:.3f}", ha="center", fontsize=9)
    ax.grid(alpha=0.3, axis="y")
    fig.tight_layout()
    path = "results/figures/e2_chemistry.png"
    fig.savefig(path)
    made.append(path)
    plt.close(fig)

if isinstance(E3, dict) and E3.get("low") and E3.get("high"):
    props = E3.get("properties", [])
    lo = [E3["low"].get(f"mean_{p}") for p in props]
    hi = [E3["high"].get(f"mean_{p}") for p in props]
    if props and all(v is not None for v in lo + hi):
        fig, axes = plt.subplots(1, len(props), figsize=(4 * len(props), 3.5), dpi=150)
        axes = [axes] if len(props) == 1 else list(axes)
        for a, p, low_v, high_v in zip(axes, props, lo, hi):
            a.bar(["low\ntarget", "high\ntarget"], [low_v, high_v],
                  color=["#4C72B0", "#C44E52"])
            a.set_title(p)
            a.grid(alpha=0.3, axis="y")
        fig.suptitle("E3 - property steering")
        fig.tight_layout()
        path = "results/figures/e3_property_steering.png"
        fig.savefig(path)
        made.append(path)
        plt.close(fig)

print("figures:")
for path in made:
    print("  ", path)
for path in made:
    try:
        from IPython.display import Image, display

        display(Image(filename=path))
    except Exception:
        pass


In [ ]:
#@title 11.3 Build the export bundle, sync to Drive, and download
import hashlib
import shutil
import tarfile
import time
from pathlib import Path

stamp = time.strftime("%Y%m%d_%H%M%S")
bundle = Path(f"geneuclidean_chemlm_v6_{stamp}.tar.gz")

candidates = [
    Path("results/tables/chemlm_v6_notebook_report.json"),
    Path("results/tables/chemlm_notebook_tier4.json"),
    Path("data/processed/crossdocked_split.json"),
]
include = [p for p in candidates if p.exists()]
# Only the figures THIS notebook produced (cell 11.2 records them in `made`).
# Globbing results/figures/*.png would sweep in every figure from earlier work.
include += [Path(p) for p in globals().get("made", []) if Path(p).exists()]

with tarfile.open(bundle, "w:gz") as tar:
    for path in include:
        tar.add(path, arcname=str(path))

print(f"bundle: {bundle} ({bundle.stat().st_size / 1e6:.2f} MB)")
print(f"sha256: {hashlib.sha256(bundle.read_bytes()).hexdigest()}")
for path in include:
    print("  ", path)

if DRIVE_DIR:
    shutil.copy2(bundle, Path(DRIVE_DIR) / bundle.name)
    print(f"\ncopied to {DRIVE_DIR}/{bundle.name}")

if IN_COLAB:
    try:
        from google.colab import files

        files.download(str(bundle))
    except Exception as exc:
        print(f"auto-download unavailable ({exc}); grab it from the Files pane")
print("\nDONE")
